In [1]:
!pip install --upgrade --quiet langchain-google-genai langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.5/471.5 kB 29.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.9.0 which is incompatible.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.5 which is incompatible.


# Remember, setup your free API Key using Google's AI Studio

https://aistudio.google.com/


In [2]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Access the API key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Initialize the model
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

## Zero-Shot Prompting

Zero-shot prompting is the most basic form of interaction with an LLM. Its performance relies entirely on its pre-trained knowledge.

In "zero-shot," the word "shot" refers to the number of examples given to the AI model, so "zero-shot" means zero examples are provided.


In [3]:
# The model is asked to solve the problem without any examples.
prompt = "Question: I started with 5 apples, bought 3 more, and then ate 2. How many do I have left?"

In [4]:
completion = llm.invoke(prompt)
print(completion.content)

ChatGoogleGenerativeAIError: Invalid argument provided to Gemini: 400 API Key not found. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API Key not found. Please pass a valid API key."
]

The model provides a direct answer, "6 apples," which is correct. However, the output lacks any reasoning or explanation. For more complex problems, this method is unreliable and can easily produce incorrect results without any way to debug the model's logic.

## Few-Shot Prompting

This technique provides the model with a few examples ("shots") to demonstrate the desired task and output format. This helps guide the model toward a more accurate response by showing it the pattern to follow.

In [ ]:
# We provide examples of other math problems to guide the model.
prompt = """
Question: I had 10 pencils and gave 4 away. How many are left?
Answer: 6

Question: I bought 2 books on Monday and 5 on Tuesday. How many books did I buy?
Answer: 7

Question: I started with 5 apples, bought 3 more, and then ate 2. How many do I have left?
"""

In [ ]:
completion = llm.invoke(prompt)
print(completion.content)

The output is again the correct number, "6". By providing examples, we increase the likelihood of getting the correct numerical answer and can better control the output format (e.g., "Answer: [number]"). However, it still doesn't reveal the underlying reasoning process, which is a significant drawback for complex tasks.

# Role-Based Prompting

Here, we assign a specific role or persona to the model to influence its tone, style, and even its problem-solving approach.

In [ ]:
# We instruct the model to act as a specific persona.
prompt = """
Act as a math tutor explaining the solution to a student.
Question: I started with 5 apples, bought 3 more, and then ate 2. How many do I have left?
"""

In [ ]:
completion = llm.invoke(prompt)
print(completion.content)

# Chain-of-Thought (CoT) Prompting

This technique explicitly instructs the model to break down the problem into intermediate steps before giving the final answer. This is often triggered by adding a simple phrase like "Let's think step-by-step".

In [ ]:
# We append a phrase that triggers step-by-step reasoning.
prompt = """
Question: I started with 5 apples, bought 3 more, and then ate 2. How many do I have left?
Let's think step-by-step.
"""

In [ ]:
completion = llm.invoke(prompt)
print(completion.content)

The output is now highly structured and transparent. The model externalizes its reasoning process, showing each calculation explicitly. This is a major improvement for complex reasoning tasks, as it allows a user to verify the logic and easily spot any errors in the process.

# Prompt Chaining

Prompt chaining is a technique where a complex task is broken down into a series of smaller, sequential prompts. The output from one prompt is used as the input for the next, creating a "chain" that guides the model through a multi-step process.

Instead of asking the model to solve the entire problem in a single, complex request, you guide it through each logical step one by one. This approach enhances reliability, control, and transparency, as you can verify the output at each stage of the process.

In [ ]:
# --- Prompt 1: First subtask (Addition) ---
prompt_1 = "I started with 5 apples and bought 3 more. How many apples do I have now?"
completion_1 = llm.invoke(prompt_1)
print(completion_1.content)

In [ ]:
# --- Prompt 2: Second subtask (Subtraction) ---
prompt_2 = f"Following the previous calculation, I now have {completion_1.content.split()[-2]} apples and ate 2. How many do I have left?"
completion_2 = llm.invoke(prompt_2)
print(completion_2.content)

Prompt chaining provides the most explicit and controlled workflow.
- **Improvement:** This method is highly reliable because each step is simple and isolated. If an error occurs, it's easy to pinpoint exactly which subtask failed, making debugging much simpler than with a single, complex prompt. It gives the developer maximum control over the reasoning process.
- **Degradation:** The primary drawback is the increased complexity and latency. This approach requires multiple calls to the model, which is slower and can be more expensive than a single prompt. It also requires more effort from the developer to break down the task and manage the flow of information between prompts. For a simple problem like this, prompt chaining is overkill, but for complex, multi-stage tasks like data analysis or campaign planning, its benefits are significant.

# ReAct (Reasoning + Acting) Prompting

ReAct (**Re**asoning and **Act**ing) represents a paradigm shift in how LLMs can solve problems. It combines the internal reasoning of Chain of Thought with the ability to take Actions and process Observations from external tools or APIs. This creates a powerful feedback loop: the model **Thinks** about what it needs to know, performs an **Action** (like a search query or API call) to get that information, receives an **Observation** (the result of the action), and then uses that new information in its next step.

The primary benefit of this approach is its ability to ground the model's reasoning in external, factual, and up-to-date information.

This `Thought -> Action -> Observation` cycle is more than just a prompting technique; it is the fundamental foundation of an agentic AI.

Let's take a more complex example to look at ReAct Prompting.

In [ ]:
# Mock function to simulate an external API call
def lookup_order_status(order_id: str) -> str:
    """Simulates looking up an order status from a database or API."""
    print(f"")
    statuses = {
        "ORD-12345": "Your order has been shipped and is expected to arrive on July 5th, 2025.",
        "ORD-67890": "Your order is currently being processed and has not yet shipped.",
        "ORD-11223": "We could not find an order with that ID. Please double-check the number."
    }
    return statuses.get(order_id, "Invalid order ID.")

In [ ]:
# A new customer email requiring external tool use
customer_email = "Hi, I'm writing to check on the status of my recent order, #ORD-12345. Can you let me know where it is?"

# The ReAct prompt needs to be manually processed in a loop for this demonstration.
# Step 1: Simulated Initial Thought
react_prompt_1 = f"""
    You have access to a tool: `lookup_order_status(order_id: str)`.
    Given the user's query, decide if you need to use the tool.
    If so, respond with the tool call. If not, respond to the user directly.

    User Query: "{customer_email}"
    Thought: The user is asking for the status of order #ORD-12345. I need to use the `lookup_order_status` tool to get this information.
    Action: lookup_order_status(order_id='ORD-12345')
"""


# Step 2: Execute the Action and get the Observation
# In a real agent, this would be automated. Here, we call the function manually.
observation = lookup_order_status(order_id='ORD-12345')
print(f"\nObservation: {observation}")


# Step 3: Use the Observation to generate the final answer
react_prompt_2 = f"""
    You have performed an action and received an observation.
    Now, formulate the final response to the user.

    User Query: "{customer_email}"
    Action Taken: lookup_order_status(order_id='ORD-12345')
    Observation: "{observation}"

    Thought: The tool returned that the order has been shipped and provided a delivery date. I should now convey this information clearly and helpfully to the customer.
    Final Answer:
"""

final_response = llm.invoke(react_prompt_2)
print(final_response.content)